In [ ]:
# MOUNT DRIVE AND CHECK GPU
from google.colab import drive
drive.mount("/content/drive")

import os
import sys
import shutil
import random
import json
import math
from pathlib import Path

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("GPU not available. Go to Runtime > Change runtime type > T4 GPU")

Mounted at /content/drive
PyTorch version: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4


In [ ]:
# V2 FOLDERS IN DRIVE
PROJECT_DIR = Path("/content/drive/MyDrive/PointNet_APS_Project_V2")

DATA_DIR = PROJECT_DIR / "data"
RAW_DATA_DIR = DATA_DIR / "raw"
GENERATED_DATA_DIR = DATA_DIR / "generated"

SRC_DIR = PROJECT_DIR / "src"
NOTEBOOKS_DIR = PROJECT_DIR / "notebooks"
MODELS_DIR = PROJECT_DIR / "models_final"
RESULTS_DIR = PROJECT_DIR / "results_final"
FIGURES_DIR = PROJECT_DIR / "figures_final"
METADATA_DIR = PROJECT_DIR / "metadata"

folders = [
    PROJECT_DIR,
    DATA_DIR,
    RAW_DATA_DIR,
    GENERATED_DATA_DIR,
    SRC_DIR,
    NOTEBOOKS_DIR,
    MODELS_DIR,
    RESULTS_DIR,
    FIGURES_DIR,
    METADATA_DIR,

    MODELS_DIR / "clean_trained",
    MODELS_DIR / "error_trained",
    MODELS_DIR / "mixed_trained",

    RESULTS_DIR / "run_logs",
    RESULTS_DIR / "test_results",
    RESULTS_DIR / "confusion_matrices",
    RESULTS_DIR / "robustness_tests",
    RESULTS_DIR / "statistics",

    FIGURES_DIR / "boxplots",
    FIGURES_DIR / "accuracy_curves",
    FIGURES_DIR / "confusion_matrices",
    FIGURES_DIR / "prediction_visualisations",
]

for folder in folders:
    folder.mkdir(parents=True, exist_ok=True)

print("V2 project folders created successfully.")
print("Project folder:", PROJECT_DIR)

V2 project folders created successfully.
Project folder: /content/drive/MyDrive/PointNet_APS_Project_V2


In [ ]:
# COPY APS DATA
APS_REPO_DIR = RAW_DATA_DIR / "BA-Primitive_Fitting_Dataset"

OLD_APS_REPO_DIR = Path("/content/drive/MyDrive/PointNet_APS_Project/BA-Primitive_Fitting_Dataset")

if APS_REPO_DIR.exists():
    print("APS dataset already exists in V2 folder:")
    print(APS_REPO_DIR)

elif OLD_APS_REPO_DIR.exists():
    print("Copying APS dataset from old project folder...")
    shutil.copytree(OLD_APS_REPO_DIR, APS_REPO_DIR)
    print("Copied APS dataset to V2 folder:")
    print(APS_REPO_DIR)

else:
    print("Old APS dataset not found. Cloning from GitHub...")
    !git clone https://github.com/lucabaronti/BA-Primitive_Fitting_Dataset.git /content/BA-Primitive_Fitting_Dataset
    shutil.copytree("/content/BA-Primitive_Fitting_Dataset", APS_REPO_DIR)
    print("Cloned and copied APS dataset to:")
    print(APS_REPO_DIR)

SHAPES_DIR = APS_REPO_DIR / "shapes"

print("\nShapes folder:")
print(SHAPES_DIR)
print("Exists:", SHAPES_DIR.exists())

APS dataset already exists in V2 folder:
/content/drive/MyDrive/PointNet_APS_Project_V2/data/raw/BA-Primitive_Fitting_Dataset

Shapes folder:
/content/drive/MyDrive/PointNet_APS_Project_V2/data/raw/BA-Primitive_Fitting_Dataset/shapes
Exists: True


In [ ]:
# Check APS dataset counts
CLASSES = ["boxes", "cylinders", "spheres"]
LABEL_MAP = {"boxes": 0, "cylinders": 1, "spheres": 2}
LABEL_NAME = {0: "box", 1: "cylinder", 2: "sphere"}

RAW_VARIANTS = ["clean", "error", "error_double"]

raw_counts = []

for variant in RAW_VARIANTS:
    for class_name in CLASSES:
        class_dir = SHAPES_DIR / variant / class_name
        files = sorted(class_dir.glob("*.csv"))

        raw_counts.append({
            "variant": variant,
            "class_folder": class_name,
            "label": LABEL_MAP[class_name],
            "count": len(files)
        })

raw_counts_df = pd.DataFrame(raw_counts)

display(raw_counts_df)

print("\nTotal per variant:")
display(raw_counts_df.groupby("variant")["count"].sum())

,variant,class_folder,label,count
0,clean,boxes,0,220
1,clean,cylinders,1,190
2,clean,spheres,2,181
3,error,boxes,0,220
4,error,cylinders,1,190
5,error,spheres,2,181
6,error_double,boxes,0,220
7,error_double,cylinders,1,190
8,error_double,spheres,2,181



Total per variant:


,count
variant,
clean,591
error,591
error_double,591


In [ ]:
# Inspect one point-cloud sample
sample_file = sorted((SHAPES_DIR / "clean" / "boxes").glob("*.csv"))[0]
sample = np.loadtxt(sample_file, delimiter=",")

print("Sample file:")
print(sample_file)

print("\nSample shape:")
print(sample.shape)

print("\nFirst 5 rows:")
print(sample[:5])

print("\nColumn meaning:")
print("Columns 0,1,2 = x, y, z coordinates")
print("Columns 3,4,5 = nx, ny, nz normal vectors")
print("Column 6 = unused/dummy column")

Sample file:
/content/drive/MyDrive/PointNet_APS_Project_V2/data/raw/BA-Primitive_Fitting_Dataset/shapes/clean/boxes/b_000.csv

Sample shape:
(1000, 7)

First 5 rows:
[[ 0.20767003  0.51768446  0.04179779  0.26188481  0.81964636  0.50950587
   0.        ]
 [ 0.41695997 -0.2588186   0.11187658  0.88515949 -0.41437179  0.21163332
   0.        ]
 [-0.27061138 -0.58583254  0.19685945 -0.3845894  -0.3955704   0.83403546
   0.        ]
 [ 0.3899546   0.03041401  0.73197997  0.26188481  0.81964636  0.50950587
   0.        ]
 [ 0.34440187 -0.53025281 -0.11610806  0.88515949 -0.41437179  0.21163332
   0.        ]]

Column meaning:
Columns 0,1,2 = x, y, z coordinates
Columns 3,4,5 = nx, ny, nz normal vectors
Column 6 = unused/dummy column


In [ ]:
# Utility file for dataset + model
%%writefile /content/drive/MyDrive/PointNet_APS_Project_V2/src/pointnetpp_aps_utils.py

import random
from pathlib import Path

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
from torch.utils.data import Dataset


CLASSES = ["boxes", "cylinders", "spheres"]
LABEL_MAP = {"boxes": 0, "cylinders": 1, "spheres": 2}
LABEL_NAME = {0: "box", 1: "cylinder", 2: "sphere"}


def set_global_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


def normalise_xyz(xyz):
    xyz = xyz.astype(np.float32)
    centroid = np.mean(xyz, axis=0)
    xyz = xyz - centroid
    scale = np.max(np.linalg.norm(xyz, axis=1))
    if scale > 0:
        xyz = xyz / scale
    return xyz


def random_rotation_matrix(rng):
    ax, ay, az = rng.uniform(0, 2 * np.pi, size=3)

    Rx = np.array([
        [1, 0, 0],
        [0, np.cos(ax), -np.sin(ax)],
        [0, np.sin(ax), np.cos(ax)]
    ], dtype=np.float32)

    Ry = np.array([
        [np.cos(ay), 0, np.sin(ay)],
        [0, 1, 0],
        [-np.sin(ay), 0, np.cos(ay)]
    ], dtype=np.float32)

    Rz = np.array([
        [np.cos(az), -np.sin(az), 0],
        [np.sin(az), np.cos(az), 0],
        [0, 0, 1]
    ], dtype=np.float32)

    return Rz @ Ry @ Rx


def make_point_record(xyz, normals):
    xyz = normalise_xyz(xyz)

    normals = normals.astype(np.float32)
    norm_len = np.linalg.norm(normals, axis=1, keepdims=True)
    norm_len[norm_len == 0] = 1.0
    normals = normals / norm_len

    dummy = np.zeros((xyz.shape[0], 1), dtype=np.float32)
    return np.concatenate([xyz, normals, dummy], axis=1)


def sample_box(n_points, rng):
    lx, ly, lz = rng.uniform(0.6, 2.0, size=3)

    face_areas = np.array([
        ly * lz, ly * lz,
        lx * lz, lx * lz,
        lx * ly, lx * ly
    ])
    face_probs = face_areas / face_areas.sum()
    faces = rng.choice(6, size=n_points, p=face_probs)

    xyz = np.zeros((n_points, 3), dtype=np.float32)
    normals = np.zeros((n_points, 3), dtype=np.float32)

    for i, face in enumerate(faces):
        x = rng.uniform(-lx / 2, lx / 2)
        y = rng.uniform(-ly / 2, ly / 2)
        z = rng.uniform(-lz / 2, lz / 2)

        if face == 0:
            x = lx / 2
            n = [1, 0, 0]
        elif face == 1:
            x = -lx / 2
            n = [-1, 0, 0]
        elif face == 2:
            y = ly / 2
            n = [0, 1, 0]
        elif face == 3:
            y = -ly / 2
            n = [0, -1, 0]
        elif face == 4:
            z = lz / 2
            n = [0, 0, 1]
        else:
            z = -lz / 2
            n = [0, 0, -1]

        xyz[i] = [x, y, z]
        normals[i] = n

    R = random_rotation_matrix(rng)
    xyz = xyz @ R.T
    normals = normals @ R.T

    return make_point_record(xyz, normals)


def sample_cylinder(n_points, rng):
    radius = rng.uniform(0.4, 1.0)
    height = rng.uniform(0.8, 2.2)

    side_area = 2 * np.pi * radius * height
    cap_area = np.pi * radius * radius
    probs = np.array([side_area, cap_area, cap_area])
    probs = probs / probs.sum()

    parts = rng.choice(3, size=n_points, p=probs)

    xyz = np.zeros((n_points, 3), dtype=np.float32)
    normals = np.zeros((n_points, 3), dtype=np.float32)

    for i, part in enumerate(parts):
        theta = rng.uniform(0, 2 * np.pi)

        if part == 0:
            z = rng.uniform(-height / 2, height / 2)
            x = radius * np.cos(theta)
            y = radius * np.sin(theta)
            n = [np.cos(theta), np.sin(theta), 0]
        else:
            r = radius * np.sqrt(rng.uniform(0, 1))
            x = r * np.cos(theta)
            y = r * np.sin(theta)

            if part == 1:
                z = height / 2
                n = [0, 0, 1]
            else:
                z = -height / 2
                n = [0, 0, -1]

        xyz[i] = [x, y, z]
        normals[i] = n

    R = random_rotation_matrix(rng)
    xyz = xyz @ R.T
    normals = normals @ R.T

    return make_point_record(xyz, normals)


def sample_sphere(n_points, rng):
    radius = rng.uniform(0.6, 1.2)

    u = rng.uniform(-1, 1, size=n_points)
    theta = rng.uniform(0, 2 * np.pi, size=n_points)

    x = radius * np.sqrt(1 - u ** 2) * np.cos(theta)
    y = radius * np.sqrt(1 - u ** 2) * np.sin(theta)
    z = radius * u

    xyz = np.stack([x, y, z], axis=1).astype(np.float32)
    normals = xyz / radius

    R = random_rotation_matrix(rng)
    xyz = xyz @ R.T
    normals = normals @ R.T

    return make_point_record(xyz, normals)


def generate_clean_primitive_dataset(output_root, n_per_class=200, n_points=1000, seed=100):
    output_root = Path(output_root)
    output_root.mkdir(parents=True, exist_ok=True)

    rng = np.random.default_rng(seed)

    generators = {
        "boxes": sample_box,
        "cylinders": sample_cylinder,
        "spheres": sample_sphere,
    }

    for class_folder, generator in generators.items():
        class_dir = output_root / class_folder
        class_dir.mkdir(parents=True, exist_ok=True)

        existing_files = list(class_dir.glob("*.csv"))
        if len(existing_files) >= n_per_class:
            print(f"{class_folder}: already exists, skipping generation.")
            continue

        for i in range(n_per_class):
            data = generator(n_points, rng)
            file_path = class_dir / f"{class_folder[:-1]}_{i:04d}.csv"
            np.savetxt(file_path, data, delimiter=",", fmt="%.8f")


def make_noisy_variant_from_clean(clean_root, output_root, noise_level=0.025, seed=200):
    clean_root = Path(clean_root)
    output_root = Path(output_root)
    output_root.mkdir(parents=True, exist_ok=True)

    rng = np.random.default_rng(seed)

    for class_folder in CLASSES:
        clean_class_dir = clean_root / class_folder
        output_class_dir = output_root / class_folder
        output_class_dir.mkdir(parents=True, exist_ok=True)

        clean_files = sorted(clean_class_dir.glob("*.csv"))

        for file_path in clean_files:
            data = np.loadtxt(file_path, delimiter=",").astype(np.float32)
            noisy = data.copy()

            xyz_noise = rng.uniform(
                low=-noise_level,
                high=noise_level,
                size=noisy[:, 0:3].shape
            ).astype(np.float32)

            noisy[:, 0:3] = noisy[:, 0:3] + xyz_noise

            output_path = output_class_dir / file_path.name
            np.savetxt(output_path, noisy, delimiter=",", fmt="%.8f")


def collect_dataset_dataframe(root_dir, variant, role, source):
    root_dir = Path(root_dir)
    records = []

    for class_folder in CLASSES:
        class_dir = root_dir / class_folder
        files = sorted(class_dir.glob("*.csv"))

        for file_path in files:
            records.append({
                "path": str(file_path),
                "class_folder": class_folder,
                "class_name": LABEL_NAME[LABEL_MAP[class_folder]],
                "label": LABEL_MAP[class_folder],
                "variant": variant,
                "role": role,
                "source": source,
                "file_name": file_path.name
            })

    return pd.DataFrame(records)


class APSPointCloudDataset(Dataset):
    def __init__(self, dataframe, n_points=1000, use_normals=True, normalise=True):
        self.df = dataframe.reset_index(drop=True).copy()
        self.n_points = n_points
        self.use_normals = use_normals
        self.normalise = normalise

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        path = row["path"]
        label = int(row["label"])

        data = np.loadtxt(path, delimiter=",").astype(np.float32)

        xyz = data[:, 0:3]

        if self.normalise:
            xyz = normalise_xyz(xyz)

        if self.use_normals:
            normals = data[:, 3:6]

            norm_len = np.linalg.norm(normals, axis=1, keepdims=True)
            norm_len[norm_len == 0] = 1.0
            normals = normals / norm_len

            points = np.concatenate([xyz, normals], axis=1)
        else:
            points = xyz

        if points.shape[0] >= self.n_points:
            choice = np.random.choice(points.shape[0], self.n_points, replace=False)
        else:
            choice = np.random.choice(points.shape[0], self.n_points, replace=True)

        points = points[choice, :]

        points = torch.tensor(points, dtype=torch.float32).transpose(0, 1)
        label = torch.tensor(label, dtype=torch.long)

        return points, label


def square_distance(src, dst):
    B, N, _ = src.shape
    _, M, _ = dst.shape

    dist = -2 * torch.matmul(src, dst.permute(0, 2, 1))
    dist += torch.sum(src ** 2, dim=-1).view(B, N, 1)
    dist += torch.sum(dst ** 2, dim=-1).view(B, 1, M)

    return dist


def index_points(points, idx):
    device = points.device
    B = points.shape[0]

    view_shape = list(idx.shape)
    view_shape[1:] = [1] * (len(view_shape) - 1)

    repeat_shape = list(idx.shape)
    repeat_shape[0] = 1

    batch_indices = torch.arange(B, dtype=torch.long, device=device).view(view_shape).repeat(repeat_shape)

    return points[batch_indices, idx, :]


def farthest_point_sample(xyz, npoint):
    device = xyz.device
    B, N, _ = xyz.shape

    centroids = torch.zeros(B, npoint, dtype=torch.long, device=device)
    distance = torch.ones(B, N, device=device) * 1e10

    farthest = torch.randint(0, N, (B,), dtype=torch.long, device=device)
    batch_indices = torch.arange(B, dtype=torch.long, device=device)

    for i in range(npoint):
        centroids[:, i] = farthest
        centroid = xyz[batch_indices, farthest, :].view(B, 1, 3)

        dist = torch.sum((xyz - centroid) ** 2, dim=-1)
        mask = dist < distance
        distance[mask] = dist[mask]

        farthest = torch.max(distance, dim=-1)[1]

    return centroids


def query_ball_point(radius, nsample, xyz, new_xyz):
    device = xyz.device
    B, N, _ = xyz.shape
    _, S, _ = new_xyz.shape

    group_idx = torch.arange(N, dtype=torch.long, device=device).view(1, 1, N).repeat(B, S, 1)

    sqrdists = square_distance(new_xyz, xyz)
    group_idx[sqrdists > radius ** 2] = N

    group_idx = group_idx.sort(dim=-1)[0][:, :, :nsample]

    group_first = group_idx[:, :, 0].view(B, S, 1).repeat(1, 1, nsample)
    mask = group_idx == N
    group_idx[mask] = group_first[mask]

    return group_idx


def sample_and_group(npoint, radius, nsample, xyz, points):
    B, N, C = xyz.shape

    fps_idx = farthest_point_sample(xyz, npoint)
    new_xyz = index_points(xyz, fps_idx)

    idx = query_ball_point(radius, nsample, xyz, new_xyz)
    grouped_xyz = index_points(xyz, idx)

    grouped_xyz_norm = grouped_xyz - new_xyz.view(B, npoint, 1, C)

    if points is not None:
        grouped_points = index_points(points, idx)
        new_points = torch.cat([grouped_xyz_norm, grouped_points], dim=-1)
    else:
        new_points = grouped_xyz_norm

    return new_xyz, new_points


def sample_and_group_all(xyz, points):
    device = xyz.device
    B, N, C = xyz.shape

    new_xyz = torch.zeros(B, 1, C, device=device)
    grouped_xyz = xyz.view(B, 1, N, C)

    if points is not None:
        grouped_points = points.view(B, 1, N, -1)
        new_points = torch.cat([grouped_xyz, grouped_points], dim=-1)
    else:
        new_points = grouped_xyz

    return new_xyz, new_points


class PointNetSetAbstraction(nn.Module):
    def __init__(self, npoint, radius, nsample, in_channel, mlp, group_all):
        super().__init__()

        self.npoint = npoint
        self.radius = radius
        self.nsample = nsample
        self.group_all = group_all

        self.mlp_convs = nn.ModuleList()
        self.mlp_bns = nn.ModuleList()

        last_channel = in_channel

        for out_channel in mlp:
            self.mlp_convs.append(nn.Conv2d(last_channel, out_channel, 1))
            self.mlp_bns.append(nn.BatchNorm2d(out_channel))
            last_channel = out_channel

    def forward(self, xyz, points):
        xyz = xyz.permute(0, 2, 1)

        if points is not None:
            points = points.permute(0, 2, 1)

        if self.group_all:
            new_xyz, new_points = sample_and_group_all(xyz, points)
        else:
            new_xyz, new_points = sample_and_group(
                self.npoint,
                self.radius,
                self.nsample,
                xyz,
                points
            )

        new_points = new_points.permute(0, 3, 2, 1)

        for conv, bn in zip(self.mlp_convs, self.mlp_bns):
            new_points = torch.relu(bn(conv(new_points)))

        new_points = torch.max(new_points, 2)[0]

        new_xyz = new_xyz.permute(0, 2, 1)

        return new_xyz, new_points


class PointNetPPClassifier(nn.Module):
    def __init__(self, num_classes=3, normal_channel=True):
        super().__init__()

        self.normal_channel = normal_channel

        self.sa1 = PointNetSetAbstraction(
            npoint=128,
            radius=0.30,
            nsample=32,
            in_channel=6 if normal_channel else 3,
            mlp=[64, 64, 128],
            group_all=False
        )

        self.sa2 = PointNetSetAbstraction(
            npoint=32,
            radius=0.60,
            nsample=64,
            in_channel=128 + 3,
            mlp=[128, 128, 256],
            group_all=False
        )

        self.sa3 = PointNetSetAbstraction(
            npoint=None,
            radius=None,
            nsample=None,
            in_channel=256 + 3,
            mlp=[256, 512, 1024],
            group_all=True
        )

        self.fc1 = nn.Linear(1024, 512)
        self.bn1 = nn.BatchNorm1d(512)
        self.drop1 = nn.Dropout(0.4)

        self.fc2 = nn.Linear(512, 256)
        self.bn2 = nn.BatchNorm1d(256)
        self.drop2 = nn.Dropout(0.4)

        self.fc3 = nn.Linear(256, num_classes)

    def forward(self, x):
        if self.normal_channel:
            xyz = x[:, 0:3, :]
            normals = x[:, 3:6, :]
        else:
            xyz = x[:, 0:3, :]
            normals = None

        l1_xyz, l1_points = self.sa1(xyz, normals)
        l2_xyz, l2_points = self.sa2(l1_xyz, l1_points)
        l3_xyz, l3_points = self.sa3(l2_xyz, l2_points)

        x = l3_points.view(l3_points.size(0), 1024)

        x = self.drop1(torch.relu(self.bn1(self.fc1(x))))
        x = self.drop2(torch.relu(self.bn2(self.fc2(x))))
        x = self.fc3(x)

        return x

Writing /content/drive/MyDrive/PointNet_APS_Project_V2/src/pointnetpp_aps_utils.py


In [ ]:
# Generate validation and test datasets
import sys
sys.path.append(str(SRC_DIR))

from pointnetpp_aps_utils import (
    generate_clean_primitive_dataset,
    make_noisy_variant_from_clean,
    collect_dataset_dataframe,
    set_global_seed
)

set_global_seed(42)

VAL_CLEAN_DIR = GENERATED_DATA_DIR / "aps_clean_val_style"
TEST_CLEAN_DIR = GENERATED_DATA_DIR / "aps_clean_test_style"
TEST_ERROR_DIR = GENERATED_DATA_DIR / "aps_error_test_style"
TEST_ERROR_DOUBLE_DIR = GENERATED_DATA_DIR / "aps_error_double_test_style"

generate_clean_primitive_dataset(
    output_root=VAL_CLEAN_DIR,
    n_per_class=200,
    n_points=1000,
    seed=100
)

generate_clean_primitive_dataset(
    output_root=TEST_CLEAN_DIR,
    n_per_class=200,
    n_points=1000,
    seed=200
)

make_noisy_variant_from_clean(
    clean_root=TEST_CLEAN_DIR,
    output_root=TEST_ERROR_DIR,
    noise_level=0.025,
    seed=300
)

make_noisy_variant_from_clean(
    clean_root=TEST_CLEAN_DIR,
    output_root=TEST_ERROR_DOUBLE_DIR,
    noise_level=0.050,
    seed=400
)

print("Validation and test datasets prepared.")

print("Validation clean:", VAL_CLEAN_DIR)
print("Test clean:", TEST_CLEAN_DIR)
print("Test error:", TEST_ERROR_DIR)
print("Test error double:", TEST_ERROR_DOUBLE_DIR)

Validation and test datasets prepared.
Validation clean: /content/drive/MyDrive/PointNet_APS_Project_V2/data/generated/aps_clean_val_style
Test clean: /content/drive/MyDrive/PointNet_APS_Project_V2/data/generated/aps_clean_test_style
Test error: /content/drive/MyDrive/PointNet_APS_Project_V2/data/generated/aps_error_test_style
Test error double: /content/drive/MyDrive/PointNet_APS_Project_V2/data/generated/aps_error_double_test_style


In [ ]:
# Build and save dataset manifest
from pointnetpp_aps_utils import collect_dataset_dataframe

raw_clean_train_df = collect_dataset_dataframe(
    root_dir=SHAPES_DIR / "clean",
    variant="aps_clean_raw",
    role="train_clean_condition",
    source="raw_github_aps"
)

raw_error_train_df = collect_dataset_dataframe(
    root_dir=SHAPES_DIR / "error",
    variant="aps_error_raw",
    role="train_error_condition",
    source="raw_github_aps"
)

raw_error_double_df = collect_dataset_dataframe(
    root_dir=SHAPES_DIR / "error_double",
    variant="aps_error_double_raw",
    role="optional_reference_only",
    source="raw_github_aps"
)

val_clean_df = collect_dataset_dataframe(
    root_dir=VAL_CLEAN_DIR,
    variant="aps_clean_val_style",
    role="validation",
    source="generated_v2"
)

test_clean_df = collect_dataset_dataframe(
    root_dir=TEST_CLEAN_DIR,
    variant="aps_clean_test_style",
    role="test_clean",
    source="generated_v2"
)

test_error_df = collect_dataset_dataframe(
    root_dir=TEST_ERROR_DIR,
    variant="aps_error_test_style",
    role="test_error",
    source="generated_v2"
)

test_error_double_df = collect_dataset_dataframe(
    root_dir=TEST_ERROR_DOUBLE_DIR,
    variant="aps_error_double_test_style",
    role="test_error_double",
    source="generated_v2"
)

manifest_df = pd.concat([
    raw_clean_train_df,
    raw_error_train_df,
    raw_error_double_df,
    val_clean_df,
    test_clean_df,
    test_error_df,
    test_error_double_df
], ignore_index=True)

manifest_path = METADATA_DIR / "dataset_manifest_v2.csv"
manifest_df.to_csv(manifest_path, index=False)

print("Dataset manifest saved to:")
print(manifest_path)

print("\nTotal count by role and variant:")
display(
    manifest_df.groupby(["role", "variant"])
    .size()
    .reset_index(name="total")
)

print("\nClass counts:")
display(
    manifest_df.groupby(["role", "variant", "class_name"])
    .size()
    .reset_index(name="count")
)

Dataset manifest saved to:
/content/drive/MyDrive/PointNet_APS_Project_V2/metadata/dataset_manifest_v2.csv

Total count by role and variant:


,role,variant,total
0,optional_reference_only,aps_error_double_raw,591
1,test_clean,aps_clean_test_style,600
2,test_error,aps_error_test_style,600
3,test_error_double,aps_error_double_test_style,600
4,train_clean_condition,aps_clean_raw,591
5,train_error_condition,aps_error_raw,591
6,validation,aps_clean_val_style,600



Class counts:


,role,variant,class_name,count
0,optional_reference_only,aps_error_double_raw,box,220
1,optional_reference_only,aps_error_double_raw,cylinder,190
2,optional_reference_only,aps_error_double_raw,sphere,181
3,test_clean,aps_clean_test_style,box,200
4,test_clean,aps_clean_test_style,cylinder,200
5,test_clean,aps_clean_test_style,sphere,200
6,test_error,aps_error_test_style,box,200
7,test_error,aps_error_test_style,cylinder,200
8,test_error,aps_error_test_style,sphere,200
9,test_error_double,aps_error_double_test_style,box,200


In [ ]:
# Check dataset sizes
print("Training clean samples:", len(raw_clean_train_df))
print("Training error samples:", len(raw_error_train_df))
print("Raw error_double samples:", len(raw_error_double_df))

print("Validation clean samples:", len(val_clean_df))
print("Test clean samples:", len(test_clean_df))
print("Test error samples:", len(test_error_df))
print("Test error_double samples:", len(test_error_double_df))

print("\nValidation class counts:")
print(val_clean_df["class_name"].value_counts())

print("\nTest clean class counts:")
print(test_clean_df["class_name"].value_counts())

print("\nTest error class counts:")
print(test_error_df["class_name"].value_counts())

print("\nTest error_double class counts:")
print(test_error_double_df["class_name"].value_counts())

Training clean samples: 591
Training error samples: 591
Raw error_double samples: 591
Validation clean samples: 600
Test clean samples: 600
Test error samples: 600
Test error_double samples: 600

Validation class counts:
class_name
box         200
cylinder    200
sphere      200
Name: count, dtype: int64

Test clean class counts:
class_name
box         200
cylinder    200
sphere      200
Name: count, dtype: int64

Test error class counts:
class_name
box         200
cylinder    200
sphere      200
Name: count, dtype: int64

Test error_double class counts:
class_name
box         200
cylinder    200
sphere      200
Name: count, dtype: int64


In [ ]:
# Checking separation (no overlap)
def path_set(df):
    return set(df["path"].astype(str))

def class_filename_set(df):
    return set((df["class_folder"].astype(str) + "/" + df["file_name"].astype(str)).tolist())

train_clean_paths = path_set(raw_clean_train_df)
train_error_paths = path_set(raw_error_train_df)

val_paths = path_set(val_clean_df)
test_clean_paths = path_set(test_clean_df)
test_error_paths = path_set(test_error_df)
test_error_double_paths = path_set(test_error_double_df)

print("Full path overlap check:")
print("Train clean vs validation:", len(train_clean_paths.intersection(val_paths)))
print("Train clean vs test clean:", len(train_clean_paths.intersection(test_clean_paths)))
print("Train clean vs test error:", len(train_clean_paths.intersection(test_error_paths)))
print("Train clean vs test error_double:", len(train_clean_paths.intersection(test_error_double_paths)))

print("Train error vs validation:", len(train_error_paths.intersection(val_paths)))
print("Train error vs test clean:", len(train_error_paths.intersection(test_clean_paths)))
print("Train error vs test error:", len(train_error_paths.intersection(test_error_paths)))
print("Train error vs test error_double:", len(train_error_paths.intersection(test_error_double_paths)))

print("\nClass + filename overlap check:")
train_clean_ids = class_filename_set(raw_clean_train_df)
train_error_ids = class_filename_set(raw_error_train_df)

val_ids = class_filename_set(val_clean_df)
test_clean_ids = class_filename_set(test_clean_df)
test_error_ids = class_filename_set(test_error_df)
test_error_double_ids = class_filename_set(test_error_double_df)

print("Train clean vs validation:", len(train_clean_ids.intersection(val_ids)))
print("Train clean vs test clean:", len(train_clean_ids.intersection(test_clean_ids)))
print("Train error vs validation:", len(train_error_ids.intersection(val_ids)))
print("Train error vs test clean:", len(train_error_ids.intersection(test_clean_ids)))

Full path overlap check:
Train clean vs validation: 0
Train clean vs test clean: 0
Train clean vs test error: 0
Train clean vs test error_double: 0
Train error vs validation: 0
Train error vs test clean: 0
Train error vs test error: 0
Train error vs test error_double: 0

Class + filename overlap check:
Train clean vs validation: 0
Train clean vs test clean: 0
Train error vs validation: 0
Train error vs test clean: 0


In [ ]:
# Create dataloaders and testing one batch
from torch.utils.data import DataLoader
from pointnetpp_aps_utils import APSPointCloudDataset

BATCH_SIZE = 8
N_POINTS = 1000
USE_NORMALS = True

train_clean_dataset = APSPointCloudDataset(
    raw_clean_train_df,
    n_points=N_POINTS,
    use_normals=USE_NORMALS
)

val_clean_dataset = APSPointCloudDataset(
    val_clean_df,
    n_points=N_POINTS,
    use_normals=USE_NORMALS
)

test_clean_dataset = APSPointCloudDataset(
    test_clean_df,
    n_points=N_POINTS,
    use_normals=USE_NORMALS
)

train_clean_loader = DataLoader(
    train_clean_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=2,
    pin_memory=torch.cuda.is_available()
)

val_clean_loader = DataLoader(
    val_clean_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=torch.cuda.is_available()
)

test_clean_loader = DataLoader(
    test_clean_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=2,
    pin_memory=torch.cuda.is_available()
)

points, labels = next(iter(train_clean_loader))

print("Batch points shape:", points.shape)
print("Batch labels shape:", labels.shape)
print("Labels:", labels[:10])

print("\nExpected points shape:")
print("[batch_size, 6, 1000]")

Batch points shape: torch.Size([8, 6, 1000])
Batch labels shape: torch.Size([8])
Labels: tensor([0, 1, 0, 1, 1, 1, 0, 2])

Expected points shape:
[batch_size, 6, 1000]


In [ ]:
# Test PointNet++
from pointnetpp_aps_utils import PointNetPPClassifier

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

model = PointNetPPClassifier(
    num_classes=3,
    normal_channel=True
).to(device)

model.eval()

points = points.to(device)

with torch.no_grad():
    logits = model(points)

print("Input shape:", points.shape)
print("Output logits shape:", logits.shape)
print("Output logits first sample:")
print(logits[0])

probs = torch.softmax(logits, dim=1)
preds = torch.argmax(probs, dim=1)

print("\nPredicted classes:", preds.cpu().numpy())
print("Confidence scores:", torch.max(probs, dim=1)[0].detach().cpu().numpy())

Using device: cuda
Input shape: torch.Size([8, 6, 1000])
Output logits shape: torch.Size([8, 3])
Output logits first sample:
tensor([-0.0162,  0.0404, -0.0059], device='cuda:0')

Predicted classes: [1 1 1 1 1 1 1 1]
Confidence scores: [0.34484947 0.344843   0.3448754  0.34490308 0.3448928  0.3448922
 0.34488562 0.3449252 ]


In [ ]:
# NB 1 STATUS
setup_status = {
    "project_dir": str(PROJECT_DIR),
    "raw_dataset_dir": str(APS_REPO_DIR),
    "shapes_dir": str(SHAPES_DIR),
    "manifest_path": str(manifest_path),
    "train_clean_samples": int(len(raw_clean_train_df)),
    "train_error_samples": int(len(raw_error_train_df)),
    "validation_clean_samples": int(len(val_clean_df)),
    "test_clean_samples": int(len(test_clean_df)),
    "test_error_samples": int(len(test_error_df)),
    "test_error_double_samples": int(len(test_error_double_df)),
    "n_points": int(N_POINTS),
    "use_normals": bool(USE_NORMALS),
    "model": "PointNet++ based classifier",
    "input_channels": 6,
    "classes": ["box", "cylinder", "sphere"],
    "notebook": "01_setup_dataset_model.ipynb",
    "status": "setup_complete_forward_pass_successful"
}

status_path = METADATA_DIR / "notebook1_setup_status.json"

with open(status_path, "w") as f:
    json.dump(setup_status, f, indent=4)

print("Notebook 1 setup status saved to:")
print(status_path)

print("\nNotebook 1 complete.")
print("Next notebook: 02_run_experiments.ipynb")

Notebook 1 setup status saved to:
/content/drive/MyDrive/PointNet_APS_Project_V2/metadata/notebook1_setup_status.json

Notebook 1 complete.
Next notebook: 02_run_experiments.ipynb
